# 02 Feature Engineering

Goal:
- Create raw transaction features.
- Create graph aggregate features.
- Save processed feature table.

In [1]:
from pathlib import Path
import sys
sys.path.append("..")

import pandas as pd

from src.data.load_data import load_transactions, save_parquet
from src.features.tabular_features import add_basic_transaction_features
from src.features.graph_features import add_account_graph_aggregate_features
from src.features.historical_graph_features import add_historical_graph_features

In [2]:
DATA_PATH = Path("../data/raw/HI-Small_Trans.csv")
NROWS = None  # increase after first successful run

df = load_transactions(DATA_PATH, nrows=NROWS)
df = add_basic_transaction_features(df)

# Static graph aggregates, useful but may include global behavior.
df_graph = add_account_graph_aggregate_features(df)

# Leakage-safe historical graph features.
df_graph = add_historical_graph_features(df_graph)

df_graph.head()

,timestamp,from_bank,sender_account,to_bank,receiver_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,...,sender_unique_receivers,sender_n_received,sender_total_received,sender_unique_senders,receiver_n_sent,receiver_total_sent,receiver_unique_receivers,receiver_n_received,receiver_total_received,receiver_unique_senders
0,2022-09-01,1411,800C81020,2843,801727270,974.63,US Dollar,974.63,US Dollar,Cash,...,3.0,1.0,49989.43,1.0,0.0,0.00,0.0,4.0,15739.72,1.0
1,2022-09-01,318398,81392FC00,318398,81392FC00,5285.34,US Dollar,5285.34,US Dollar,Reinvestment,...,1.0,1.0,5285.34,1.0,1.0,5285.34,1.0,1.0,5285.34,1.0
2,2022-09-01,18475,80D6B5440,18475,80D6B5440,70729.11,Yuan,70729.11,Yuan,Reinvestment,...,2.0,1.0,70729.11,1.0,3.0,70856.35,2.0,1.0,70729.11,1.0
3,2022-09-01,353541,8139376B0,221731,8139349A0,788.65,US Dollar,788.65,US Dollar,ACH,...,1.0,0.0,0.00,0.0,1.0,8387.21,1.0,2.0,9175.86,2.0
4,2022-09-01,317588,8071DF720,317588,8071DF720,9819.65,Ruble,9819.65,Ruble,Reinvestment,...,1.0,1.0,9819.65,1.0,1.0,9819.65,1.0,1.0,9819.65,1.0


In [3]:
print("Processed shape:", df_graph.shape)
graph_cols = [c for c in df_graph.columns if c.startswith("sender_") or c.startswith("receiver_")]
print("Graph feature count:", len(graph_cols))
print(graph_cols[:30])

Processed shape: (500000, 33)
Graph feature count: 16
['sender_account', 'receiver_account', 'sender_id', 'receiver_id', 'sender_n_sent', 'sender_total_sent', 'sender_unique_receivers', 'sender_n_received', 'sender_total_received', 'sender_unique_senders', 'receiver_n_sent', 'receiver_total_sent', 'receiver_unique_receivers', 'receiver_n_received', 'receiver_total_received', 'receiver_unique_senders']


In [4]:
OUT_PATH = Path("../data/processed/hi_small_features.parquet")
save_parquet(df_graph, OUT_PATH)
print("Saved:", OUT_PATH)

Saved: ../data/processed/hi_small_features.parquet
